In [1]:
%pwd

'c:\\Users\\ecole\\CoreML-Ayush\\resume-projects\\kkbox-churn\\explore'

In [2]:
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

%pwd

'c:\\Users\\ecole\\CoreML-Ayush\\resume-projects\\kkbox-churn'

In [3]:
import duckdb

con = duckdb.connect()
DATA = "data"

SEP = "\n" + "=" * 70 + "\n"
SEP2 = "\n" + "-" * 50 + "\n"


def section(title):
    print(f"{SEP}{title}{SEP}")


def sub(title):
    print(f"{SEP2}{title}{SEP2}")


def q(sql):
    return con.execute(sql).df()


def show(df, max_rows=40):
    print(df.to_string(index=False, max_rows=max_rows))
    print()


# Project walkthrough

- Predict for users expiring within March 1 – March 31, 2017.
- 4 tables provided: user_logs, transactions, train (labels), members (static).
- Feature cutoff = expiry − 14 days → we know 14 days ahead whether a user will eventually churn. This is this project's own production framing (per the README's "intervene 14 days before expiry"), layered on top of KKBox's label definition — it's not something KKBox itself defines. Keep these two things mentally separate: KKBox's 30-day rule defines the label (ground truth, looking forward from expiry); the 14-day cutoff defines feature availability (a separate, earlier boundary for the production system).


user_logs.csv coverage — mostly right, mechanism was off:
- Right: it only covers March 1–31, so some users have no log data.
- Fix: it's not "we're missing a rolling 30-day window." The file has zero rows before March 1, period.

expires=march 1
label=april 1
data to use to predict label=everything before March 1
but we have only MArch

expiry is March (not Feb), and predicting April renewal 
To confirm whether a March-expiring user renews after 30days, you need transaction records into April, and transactions.csv stops at 3/31/2017.
We don't have April's transaction data, or any April data for any table sef.

it's user_logs table we don't need APril, we need past logs not future for user_logs

Your hypothesis: "Maybe these 930K users don't have a March expiry date because their plans are just longer — like, they signed up for a longer subscription, so their expiry naturally lands later than March."

What I found: I checked the actual plan lengths. Both groups — the 40K users who DO expire in March, and the 930K who don't — have the same typical plan length: 30 days. So no, it's not that the 930K bought longer plans. Plan length isn't the reason.

What actually explains it instead: these are mostly long-time subscribers who keep renewing. Every time they renew, their expiry date gets pushed forward from wherever it currently sits — not recalculated fresh from today's date. So depending on their personal renewal history, their expiry could land in March, April, May, or later. It's really just a matter of timing/luck in their own renewal cycle, not anything about the length of any one plan.

What it means, in plain terms: those 930K users genuinely are not showing a March expiry in the data — and that's not a data error or something missing. It's simply that, as of this snapshot, their subscription happens to run past March. So your original instinct was right: "no March record" mostly just means "not expiring in March," not "the record is broken or lost."

The bigger implication: train.csv is supposed to be "all the users whose subscription expires in March 2017" (that's the project's whole label cohort definition) — but only about 4% of train.csv's users actually show that in the data we have. That's a real mismatch worth sitting with: either train.csv's cohort isn't as narrowly defined as "expires in March" as we assumed, or KKBox had extra transaction history for these users that never made it into our transactions.csv extract. Either way, we can't currently tell which one it is.


#1 — correct, and not wasted effort. train.csv can't be regenerated from transactions.csv — confirmed twice over (Error 3: no April data to check renewals; Error 4: most of train.csv's cohort doesn't even show a March expiry to anchor from). The "why did I stress about this" instinct is fair for the original question (can we verify labels ourselves — no, use train.csv as-is). But the investigation wasn't wasted: it's what surfaced Error 4's deeper problem, which isn't about labels at all — it's about whether the 930K "unanchored" users' features can be trusted.


#2 — right direction, but understates how bad it is. "Some days missing, so aggregate over what we have" describes a partial gap — a user with 20 of 30 days logged, aggregate over 20. That's not what's actually happening for most rows. It's closer to all-or-nothing: 98.2% of the full training set has zero log rows in their window, not a partial window. Even restricting to the "good" 40,227 anchored users, 52.6% still have zero log rows (Cell 6). So it's not "aggregate around some missing days" — it's "has_log_data=0, entire row is NaN, LightGBM never sees an engagement signal for this user at all."


#3 — is it true? Partially, and it's actually a better instinct than it might sound like, for a reason you didn't state: every user scored in production has a real, known expiry date (per the serving design — has_anchor is always 1 at serving time). So training only on the 40,227 users with a real anchor means your training feature distribution actually matches what the model will see in production. Training on the full 970K (with the Feb-15 fallback for 96% of rows) means the model partly learns from a feature pattern — null transaction features, a fake fixed cutoff — that never occurs at serving time. That's the real mechanism behind "bad data = bad model" here: it's not vague badness, it's train/serve skew.

Problems that still suface:

1. Sample size collapses hard. 970,960 → 40,227 is a 96% cut. At ~9% churn, that's roughly 3,600 churn examples left, down from ~87,330. That's a real variance/overfitting risk, not just a cleanliness win — smaller-but-cleaner isn't automatically better.
sol: use it to train. in production this wouldn't be the case anyways

2. Restricting to anchored users doesn't fix #2. Log data is still missing for 52.6% of even this "clean" subset — that's Gap 3 (user_logs.csv only covers March).
sol: Switching to option #3 fixes the transaction/member-feature train/serve skew, very good.
don't mind the missing logs

3. Possible new selection bias. The plan-length check showed the anchored group is almost entirely 30-day-plan subscribers. If quarterly/annual-plan users are underrepresented among "anchored" users (worth checking, not yet confirmed), the model may generalize poorly to non-monthly subscribers.
sol: kkbox says "The churn/renewal definition can be tricky due to KKBox's subscription model. Since the majority of KKBox's subscription length is 30 days, a lot of users re-subscribe every month. The criteria of "churn" is no new valid service subscription within 30 days after the current membership expires."
they accept the tradeoff

4. This is a hypothesis, not yet validated. You should actually test both approaches (train on all 970K+fallback vs. train on the 40K anchored subset) and compare on a held-out slice — ideally a different month's cohort — rather than assume the smaller/cleaner set wins. "Bad data in, bad model out" is a heuristic; the real answer is empirical.
sol: yeah i know

---
train.csv-contains the churn data for March, 2017.
transactions.csv-contains the transactions data until 3/31/2017.
userlogs-contains the user logs data for MArch 2017.
submission-contains the test data for April, 2017. 

QUESTION: model runs daily to predict? what was my decision and why?

In [8]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 -- Which cohort is train.csv actually? Feb-2017 expiry or March-2017
#           expiry? The official KKBox competition had TWO rounds with
#           different answers, and it's easy to quote the wrong one:
#             Round 1: train = users expiring Feb 2017 (churn observed ~March)
#             Round 2 (_v2 files): train = users expiring March 2017
#                                  (churn observed ~April)
#           transactions.csv here runs through 2017-03-31, and train.csv has
#           exactly 970,960 rows -- both point at round 2 -- but let's not
#           take that on faith. Two direct tests below:
#             (a) row count check
#             (b) which expiry-month cohort actually overlaps train.csv,
#                 and which cohort's KKBox-rule-derived is_churn actually
#                 agrees with train.csv's real labels
# ══════════════════════════════════════════════════════════════════════════════
section("CELL 7 -- IS train.csv THE FEBRUARY OR MARCH 2017 EXPIRY COHORT?")

sub("Row count -- known dataset variants: round-1 train.csv has 992,931 rows, round-2 train_v2.csv has 970,960")
show(q(f"SELECT COUNT(*) AS train_rows FROM read_csv_auto('{DATA}/train.csv')"))

def _cohort_agreement(month_start, month_end, label):
    return q(f"""
    WITH txns AS (
      SELECT *,
        STRPTIME(CAST(transaction_date AS VARCHAR), '%Y%m%d') AS txn_dt,
        STRPTIME(CAST(membership_expire_date AS VARCHAR), '%Y%m%d') AS expire_dt
      FROM read_csv_auto('{DATA}/transactions.csv')
    ),
    cohort AS (
      SELECT msno, MAX(expire_dt) AS anchor_expiry_dt
      FROM txns
      WHERE expire_dt >= '{month_start}' AND expire_dt <= '{month_end}'
      GROUP BY msno
    ),
    renewals AS (
      SELECT DISTINCT t.msno
      FROM txns t JOIN cohort c ON t.msno = c.msno
      WHERE t.txn_dt > c.anchor_expiry_dt
        AND t.txn_dt <= c.anchor_expiry_dt + INTERVAL 30 DAY
        AND t.is_cancel = 0
    ),
    derived AS (
      SELECT c.msno, CASE WHEN r.msno IS NULL THEN 1 ELSE 0 END AS derived_churn
      FROM cohort c LEFT JOIN renewals r ON c.msno = r.msno
    ),
    train AS (SELECT * FROM read_csv_auto('{DATA}/train.csv')),
    matched AS (
      SELECT d.msno, d.derived_churn, t.is_churn AS train_churn
      FROM derived d JOIN train t ON d.msno = t.msno
    )
    SELECT
      '{label}' AS cohort,
      (SELECT COUNT(*) FROM cohort) AS total_anchored_in_extract,
      COUNT(*) AS also_in_train_csv,
      ROUND(100.0*COUNT(*) / (SELECT COUNT(*) FROM cohort), 2) AS pct_of_cohort_in_train,
      ROUND(100.0*SUM(CASE WHEN derived_churn = train_churn THEN 1 ELSE 0 END)/COUNT(*), 2) AS is_churn_agree_pct
    FROM matched
    """)

sub("Feb-2017-expiry cohort vs train.csv")
show(_cohort_agreement('2017-02-01', '2017-02-28', 'Feb-2017 expiry'))

sub("March-2017-expiry cohort vs train.csv")
show(_cohort_agreement('2017-03-01', '2017-03-31', 'Mar-2017 expiry'))

print("March wins on overlap, decisively: ~99% of identifiable March-expiry")
print("users are in train.csv, vs a handful (350 total, 25 in train) for Feb.")
print("That confirms train.csv IS the March-2017 cohort (round 2 / _v2),")
print("matching CLAUDE.md -- NOT the round-1 'train=Feb, test=March' framing")
print("that's easy to find if you read the competition's original overview.")
print()
print("The March cohort's is_churn agreement with our own derived labels is")
print("only ~57% -- that is NOT evidence against March being the right cohort.")
print("It's the direct consequence of Cell 1: ~98% of the March cohort needs")
print("April transaction data to verify their 30-day renewal, and we don't")
print("have any April data, so our self-derived labels are biased toward")
print("false 'churn' for anyone who actually renewed in April.")


CELL 7 -- IS train.csv THE FEBRUARY OR MARCH 2017 EXPIRY COHORT?


--------------------------------------------------
Row count -- known dataset variants: round-1 train.csv has 992,931 rows, round-2 train_v2.csv has 970,960
--------------------------------------------------

 train_rows
     970960


--------------------------------------------------
Feb-2017-expiry cohort vs train.csv
--------------------------------------------------

         cohort  total_anchored_in_extract  also_in_train_csv  pct_of_cohort_in_train  is_churn_agree_pct
Feb-2017 expiry                        350                 25                    7.14                84.0


--------------------------------------------------
March-2017-expiry cohort vs train.csv
--------------------------------------------------

         cohort  total_anchored_in_extract  also_in_train_csv  pct_of_cohort_in_train  is_churn_agree_pct
Mar-2017 expiry                      40227              39972                   99.37            

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 -- What month are we actually predicting churn for, and why is it hard?
#
# The label cohort is "users whose subscription expires in March 2017"
# (CLAUDE.md > Churn Definition). A user is is_churn=1 if no new valid
# transaction (is_cancel=0) shows up within 30 days of their expiry date.
# That 30-day window is the whole problem: to confirm a user who expires
# on, say, March 31, DIDN'T renew, you need transaction data through
# April 30. transactions.csv stops at March 31, 2017 -- so for anyone
# expiring in the back half of March, we cannot independently verify
# their own label. This cell proves the data cutoff and quantifies how
# many March-cohort users fall in that unverifiable window.
# ══════════════════════════════════════════════════════════════════════════════
section("CELL 1 -- THE CHURN MONTH IS MARCH 2017 -- WHY IT'S HARD TO VERIFY")

sub("The raw data's hard stop")
show(
    q(f"""
SELECT MAX(transaction_date) AS last_date_we_have
FROM read_csv_auto('{DATA}/transactions.csv')
""")
)
print("Every transaction record in the entire extract stops here. There is")
print("no April 2017 data at all, in this file or any other.")

sub("March-2017 expiring users, split by whether their 30-day renewal window")
sub("extends past that hard stop (2017-03-31)")
show(
    q(f"""
WITH txns AS (
  SELECT *, STRPTIME(CAST(membership_expire_date AS VARCHAR), '%Y%m%d') AS expire_dt
  FROM read_csv_auto('{DATA}/transactions.csv')
),
mar_cohort AS (
  SELECT msno, MAX(expire_dt) AS anchor_expiry_dt
  FROM txns
  WHERE expire_dt BETWEEN '2017-03-01' AND '2017-03-31'
  GROUP BY msno
)
SELECT
  COUNT(*) AS total_march_cohort,
  COUNT(*) FILTER (WHERE anchor_expiry_dt + INTERVAL '30 days' > '2017-03-31')
    AS renewal_window_runs_past_data_end,
  ROUND(100.0 * COUNT(*) FILTER (WHERE anchor_expiry_dt + INTERVAL '30 days' > '2017-03-31')
        / COUNT(*), 2) AS pct_unverifiable
FROM mar_cohort
""")
)
print("This is why CLAUDE.md says: use train.csv labels as authoritative --")
print("KKBox generated them with data we were never given (April 2017),")
print("so we cannot regenerate/verify them ourselves for most of the cohort.")



CELL 1 -- THE CHURN MONTH IS MARCH 2017 -- WHY IT'S HARD TO VERIFY


--------------------------------------------------
The raw data's hard stop
--------------------------------------------------

 last_date_we_have
          20170331

Every transaction record in the entire extract stops here. There is
no April 2017 data at all, in this file or any other.

--------------------------------------------------
March-2017 expiring users, split by whether their 30-day renewal window
--------------------------------------------------


--------------------------------------------------
extends past that hard stop (2017-03-31)
--------------------------------------------------

 total_march_cohort  renewal_window_runs_past_data_end  pct_unverifiable
              40227                              39532             98.27

This is why CLAUDE.md says: use train.csv labels as authoritative --
KKBox generated them with data we were never given (April 2017),
so we cannot regenerate/verify them our

Cell 1 shows 98.27% of the March cohort falls in this unverifiable position — which is exactly why train.csv's labels are treated as authoritative (KKBox computed them with April data we were never given) rather than something we re-derive.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. transactions.csv - date range overview
# ══════════════════════════════════════════════════════════════════════════════
section("1. TRANSACTIONS - DATE RANGE OVERVIEW")

show(
    q(f"""
SELECT
  MIN(transaction_date)       AS txn_date_min,
  MAX(transaction_date)       AS txn_date_max,
  MIN(membership_expire_date) AS expire_date_min,
  MAX(membership_expire_date) AS expire_date_max,
  COUNT(*) FILTER (WHERE transaction_date < 20150101)       AS txn_before_2015,
  COUNT(*) FILTER (WHERE transaction_date > 20170331)       AS txn_after_mar2017,
  COUNT(*) FILTER (WHERE membership_expire_date > 20991231) AS expire_far_future
FROM read_csv_auto('{DATA}/transactions.csv')
""")
)

sub("Sample of far-future expiry rows (legacy long-term accounts)")
show(
    q(f"""
SELECT msno, transaction_date, membership_expire_date, payment_plan_days, plan_list_price
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE membership_expire_date > 20991231
ORDER BY membership_expire_date DESC
LIMIT 15
""")
)



1. TRANSACTIONS - DATE RANGE OVERVIEW

 txn_date_min  txn_date_max  expire_date_min  expire_date_max  txn_before_2015  txn_after_mar2017  expire_far_future
     20150101      20170331         20160419         20361015                0                  0                  0


--------------------------------------------------
Sample of far-future expiry rows (legacy long-term accounts)
--------------------------------------------------

Empty DataFrame
Columns: [msno, transaction_date, membership_expire_date, payment_plan_days, plan_list_price]
Index: []



In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Error 2 - membership_expire_date < transaction_date (backdated records)
# ══════════════════════════════════════════════════════════════════════════════
section("2. ERROR 2 - EXPIRE DATE BEFORE TRANSACTION DATE")

show(
    q(f"""
SELECT COUNT(*) AS bad_expiry_rows
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE membership_expire_date < transaction_date
""")
)

sub("Sample of 15 offending rows")
show(
    q(f"""
SELECT msno, transaction_date, membership_expire_date, is_cancel, is_auto_renew
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE membership_expire_date < transaction_date
ORDER BY RANDOM()
LIMIT 15
""")
)



2. ERROR 2 - EXPIRE DATE BEFORE TRANSACTION DATE

 bad_expiry_rows
            5106


--------------------------------------------------
Sample of 15 offending rows
--------------------------------------------------

                                        msno  transaction_date  membership_expire_date  is_cancel  is_auto_renew
XFdENUQ1uEkKVRTNtvBxN/TIvR6VEYBW6udNr1ZfoZI=          20170303                20170302          1              1
8q6fcUBIQbkYxfdspqmuX9uhqhQ8B6qMdfbAv+ACBUI=          20170303                20170302          1              1
CFytk8Ya5xrGGtW1Fpu8HDQoYnBMltECDhNW3CxQ2QA=          20170305                20170303          1              1
YsYF8ns6BogaWMJ4V7QE+q/m/i3ASXZwtShvZxF1iV0=          20170328                20170327          1              1
c1ixbTW7vzQvYsqg/10tGznbJQyzIfTKDCauG9pU3U4=          20170308                20170307          1              1
YUyBQ2xf7PQwZbuRE+foZdH6M5Q5yZ11V3OuIzJpvHw=          20170304                20170303          1       

i caught a real overstatement — let me show you why, and dig into what's actually there.
[ what was i thinking then? i dey sleep? ]



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. Error 1 - legacy migration records
# ══════════════════════════════════════════════════════════════════════════════
section("3. ERROR 1 - LEGACY MIGRATION RECORDS")

show(
    q(f"""
SELECT COUNT(*) AS legacy_rows
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE payment_plan_days = 0 AND plan_list_price = 0 AND actual_amount_paid > 0
""")
)

sub("transaction_date distribution for legacy rows (expect clustered Apr-May 2015)")
show(
    q(f"""
SELECT transaction_date, COUNT(*) AS cnt
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE payment_plan_days = 0 AND plan_list_price = 0 AND actual_amount_paid > 0
GROUP BY 1 ORDER BY 1
""")
)

sub("membership_expire_date range for legacy rows (expect far-future skew)")
show(
    q(f"""
SELECT
  MIN(membership_expire_date) AS expire_min,
  MAX(membership_expire_date) AS expire_max,
  COUNT(*) FILTER (WHERE membership_expire_date > 20991231) AS far_future_cnt
FROM read_csv_auto('{DATA}/transactions.csv')
WHERE payment_plan_days = 0 AND plan_list_price = 0 AND actual_amount_paid > 0
""")
)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. Error 5 - future member registration dates (resolve cutoff discrepancy)
# ══════════════════════════════════════════════════════════════════════════════
section(
    "4. ERROR 5 - FUTURE REGISTRATION DATES (CLAUDE.md: >20170331 vs docs: >20170317)"
)

show(
    q(f"""
SELECT
  COUNT(*) FILTER (WHERE registration_init_time > 20170317) AS after_20170317,
  COUNT(*) FILTER (WHERE registration_init_time > 20170331) AS after_20170331,
  MIN(registration_init_time) FILTER (WHERE registration_init_time > 20170317) AS min_future_reg,
  MAX(registration_init_time) AS max_reg_overall
FROM read_csv_auto('{DATA}/members.csv')
""")
)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. Gap 1 - anchor coverage for train.csv cohort (resolve 40,227 vs 35,433)
# ══════════════════════════════════════════════════════════════════════════════
section("5. GAP 1 - ANCHOR COVERAGE FOR train.csv COHORT")

show(
    q(f"""
WITH train AS (SELECT msno FROM read_csv_auto('{DATA}/train.csv')),
     anchored AS (
       SELECT DISTINCT msno
       FROM read_csv_auto('{DATA}/transactions.csv')
       WHERE membership_expire_date BETWEEN 20170301 AND 20170331
     )
SELECT
  (SELECT COUNT(*) FROM train)                                          AS train_total,
  COUNT(DISTINCT a.msno) FILTER (WHERE t.msno IS NOT NULL)               AS anchored_in_train,
  ROUND(100.0 * COUNT(DISTINCT a.msno) FILTER (WHERE t.msno IS NOT NULL)
        / (SELECT COUNT(*) FROM train), 2)                              AS pct_covered
FROM anchored a
LEFT JOIN train t ON a.msno = t.msno
""")
)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. Gap 3 - user_logs.csv date coverage
# ══════════════════════════════════════════════════════════════════════════════
section("6. GAP 3 - USER_LOGS.CSV DATE COVERAGE")

show(
    q(f"""
SELECT
  MIN(date)            AS log_date_min,
  MAX(date)             AS log_date_max,
  COUNT(DISTINCT date) AS distinct_dates
FROM read_csv_auto('{DATA}/user_logs.csv')
""")
)
print("Expect distinct_dates = 31 (March 2017 only). Users with feature_cutoff_dt")
print("before 2017-03-01 (i.e. anchor_expiry before 2017-03-15) get zero log rows.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. Feature cutoff range for the anchored March 2017 cohort
# ══════════════════════════════════════════════════════════════════════════════
section("7. FEATURE CUTOFF RANGE - ANCHORED MARCH 2017 COHORT")

show(
    q(f"""
WITH txns AS (
  SELECT *,
    STRPTIME(CAST(membership_expire_date AS VARCHAR), '%Y%m%d') AS expire_dt
  FROM read_csv_auto('{DATA}/transactions.csv')
),
mar_cohort AS (
  SELECT msno, MAX(expire_dt) AS anchor_expiry_dt
  FROM txns
  WHERE expire_dt >= '2017-03-01' AND expire_dt <= '2017-03-31'
  GROUP BY msno
)
SELECT
  MIN(anchor_expiry_dt - INTERVAL '14 days') AS cutoff_min,
  MAX(anchor_expiry_dt - INTERVAL '14 days') AS cutoff_max,
  COUNT(*)                                    AS anchored_users
FROM mar_cohort
""")
)
print(
    "Expect range ~2017-02-15 to ~2017-03-17, matching CLAUDE.md's Label cohort section."
)

print("\nDate issues check complete.")
